In [1]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Task 1: Knowledge Base

knowledge_base = [
    "To reset your password, click on the Forgot Password link on the login page.",
    "Billing invoices can be downloaded from the Billing section in your account.",
    "You can update your email address from your profile settings.",
    "If you cannot log in, ensure your username and password are correct.",
    "Two-factor authentication improves account security.",
    "You can cancel your subscription anytime from the subscription page.",
    "Refund requests are processed within 7 business days.",
    "Contact customer support if your account has been locked.",
    "Premium users receive priority customer support.",
    "Your account information can be edited from the Account Settings page."
]

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
embeddings = model.encode(
    knowledge_base,
    convert_to_numpy=True
)

print("Embedding Shape:", embeddings.shape)

# Task 2: Build FAISS Index


dimension = embeddings.shape[1]

# Normalize embeddings
faiss.normalize_L2(embeddings)

# Create index
index = faiss.IndexFlatL2(dimension)

# Add vectors
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

# Task 3: Semantic Search

def search(query, k=3):
    query_vector = model.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(query_vector)

    distances, indices = index.search(query_vector, k)

    print("\nQuery:", query)
    print("-" * 80)
    print(f"{'Rank':<5}{'Score(L2)':<15}Matched Sentence")

    for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), start=1):
        print(f"{rank:<5}{score:<15.4f}{knowledge_base[idx]}")

# Test Queries
test_queries = [
    "How do I change my password?",
    "Where can I download my invoice?",
    "I cannot access my account."
]

for q in test_queries:
    search(q)

# Task 4: Interactive CLI


print("\nSemantic Search CLI")
print("Type 'exit' to quit.")

while True:
    query = input("\nEnter your query: ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    search(query)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 6479.88it/s]


Embedding Shape: (10, 384)
Total vectors stored: 10

Query: How do I change my password?
--------------------------------------------------------------------------------
Rank Score(L2)      Matched Sentence
1    0.6840         To reset your password, click on the Forgot Password link on the login page.
2    0.9845         Contact customer support if your account has been locked.
3    1.0560         Your account information can be edited from the Account Settings page.

Query: Where can I download my invoice?
--------------------------------------------------------------------------------
Rank Score(L2)      Matched Sentence
1    0.5138         Billing invoices can be downloaded from the Billing section in your account.
2    1.4882         Premium users receive priority customer support.
3    1.6769         You can cancel your subscription anytime from the subscription page.

Query: I cannot access my account.
-----------------------------------------------------------------------------


Enter your query:  I forgot my password



Query: I forgot my password
--------------------------------------------------------------------------------
Rank Score(L2)      Matched Sentence
1    0.5466         To reset your password, click on the Forgot Password link on the login page.
2    0.8497         Contact customer support if your account has been locked.
3    1.0084         If you cannot log in, ensure your username and password are correct.



Enter your query:  where can i find the invoice



Query: where can i find the invoice
--------------------------------------------------------------------------------
Rank Score(L2)      Matched Sentence
1    0.6603         Billing invoices can be downloaded from the Billing section in your account.
2    1.4941         Premium users receive priority customer support.
3    1.5758         Refund requests are processed within 7 business days.



Enter your query:  I can't find my account



Query: I can't find my account
--------------------------------------------------------------------------------
Rank Score(L2)      Matched Sentence
1    0.8066         Contact customer support if your account has been locked.
2    0.9154         If you cannot log in, ensure your username and password are correct.
3    1.0754         Your account information can be edited from the Account Settings page.
